In [ ]:
jogo = 22912

In [10]:
import pandas as pd
from conexao import query

sql = """
    SELECT g.team_1_name,
           g.team_2_name,
           l.player_name,
           l.team_name AS team_player,
           l.jersey_number,
           e.action,
           e.minute,
           e.pass_length,
           e.pass_outcome,
           e.shot_outcome,
           e.shot_technique,
           e.shot_xg
    FROM events e
    JOIN games g ON e.game = g.game_id
    JOIN lineups l ON g.game_id = l.game AND e.player_id = l.player_id
    WHERE g.game_id = %s
    ORDER BY e.minute, e.second
"""

rows = query(sql).param(jogo).execute()
df = pd.DataFrame(rows)


In [11]:
import pandas as pd
from conexao import query

sql = """
    SELECT
        l.player_name,
        l.jersey_number,
        l.team_name AS team_player,

        COUNT(CASE WHEN e.action = 'Pass' THEN 1 END)
            AS tentativas_passe,

        ROUND(
            COUNT(CASE WHEN e.action = 'Pass' AND e.pass_outcome IS NULL THEN 1 END)
            / NULLIF(COUNT(CASE WHEN e.action = 'Pass' THEN 1 END), 0) * 100
        , 1)
            AS perc_acerto_passe,

        COUNT(CASE WHEN e.action = 'Shot' THEN 1 END)
            AS tentativas_chute,

        COUNT(CASE WHEN e.action = 'Shot' AND e.shot_outcome != 'Off T' THEN 1 END)
            AS chutes_no_gol,

        ROUND(SUM(e.shot_xg), 3)
            AS xg_total,

        COUNT(CASE WHEN e.action = 'Ball Recovery' THEN 1 END)
            AS bolas_recuperadas,

        COUNT(CASE WHEN e.action = 'Shot' AND e.shot_outcome = 'Goal' THEN 1 END)
            AS gols,

        COUNT(DISTINCT goal_shot.id)
            AS assistencias

    FROM events e
    JOIN games g ON e.game = g.game_id
    JOIN lineups l ON g.game_id = l.game AND e.player_id = l.player_id
    LEFT JOIN events goal_shot
        ON goal_shot.key_pass_id = e.id
        AND goal_shot.game = e.game
        AND goal_shot.shot_outcome = 'Goal'
    WHERE g.game_id = %s
    GROUP BY l.player_id, l.player_name, l.jersey_number, l.team_name
    ORDER BY l.team_name, l.jersey_number
"""

stats = pd.DataFrame(query(sql).param(jogo).execute())
stats


,player_name,jersey_number,team_player,tentativas_passe,perc_acerto_passe,tentativas_chute,chutes_no_gol,xg_total,bolas_recuperadas,gols,assistencias
0,Víctor Valdés Arribas,1,Barcelona,30,53.3,0,0,NaN,5,0,0
1,Gerard Piqué Bernabéu,3,Barcelona,35,80.0,0,0,NaN,0,0,0
2,Carles Puyol i Saforcada,5,Barcelona,53,69.8,2,2,0.513,4,0,0
3,Xavier Hernández Creus,6,Barcelona,84,94.0,2,1,0.100,5,0,1
4,Andrés Iniesta Luján,8,Barcelona,71,93.0,2,2,0.079,7,0,1
5,Samuel Eto''o Fils,9,Barcelona,24,75.0,1,1,0.102,5,1,0
6,Lionel Andrés Messi Cuccittini,10,Barcelona,57,91.2,2,1,0.243,5,1,0
7,Thierry Henry,14,Barcelona,15,86.7,2,2,0.161,2,0,0
8,Seydou Kéita,15,Barcelona,13,84.6,0,0,NaN,1,0,0
9,Sylvio Mendes Campos Junior,16,Barcelona,60,81.7,0,0,NaN,3,0,0


In [12]:
from conexao import conectar

sql_insert = """
    INSERT INTO stats_player (
        game_id, player_name, jersey_number, team_name,
        tentativas_passe, perc_acerto_passe,
        tentativas_chute, chutes_no_gol,
        xg_total, bolas_recuperadas,
        gols, assistencias
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

rows = [
    (
        jogo,
        row["player_name"],
        row["jersey_number"],
        row["team_player"],
        row["tentativas_passe"],
        row["perc_acerto_passe"],
        row["tentativas_chute"],
        row["chutes_no_gol"],
        row["xg_total"],
        row["bolas_recuperadas"],
        row["gols"],
        row["assistencias"],
    )
    for _, row in stats.iterrows()
]

conn = conectar()
try:
    cursor = conn.cursor()
    cursor.executemany(sql_insert, rows)
    conn.commit()
    print(f"{cursor.rowcount} registros inseridos.")
finally:
    cursor.close()
    conn.close()


26 registros inseridos.
